In [10]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

In [11]:
spark = SparkSession.builder \
    .master("spark://localhost:7077") \
    .appName('test') \
    .getOrCreate()

In [12]:
spark

In [13]:
df_green =spark.read.parquet('C:/Tools/tmp/data/pq/green/*/*')

In [14]:
df_yellow=spark.read.parquet('C:/Tools/tmp/data/pq/yellow/*/*')

In [15]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [16]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')  


In [17]:
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)


df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))        


df_trips_data = df_green_sel.unionAll(df_yellow_sel)


In [22]:
df_trips_data.groupBy('service_type').count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [18]:
df_trips_data.registerTempTable('trips_data')

C:\tools\spark-3.5.4-bin-hadoop3\python\pyspark\sql\dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [19]:
df_result = spark.sql("""
SELECT 
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_montly_passenger_count,
    AVG(trip_distance) AS avg_montly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [20]:
df_result \
    .select('revenue_zone', 'revenue_month', 'service_type', 'revenue_monthly_total_amount', 'avg_montly_passenger_count') \
    .show()

+------------+-------------------+------------+----------------------------+--------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_total_amount|avg_montly_passenger_count|
+------------+-------------------+------------+----------------------------+--------------------------+
|           3|2020-01-01 00:00:00|       green|          14820.079999999996|        1.0987654320987654|
|           9|2020-01-01 00:00:00|       green|                    12017.93|        1.0833333333333333|
|          23|2020-01-01 00:00:00|       green|           983.1800000000001|                       2.0|
|          71|2020-01-01 00:00:00|       green|           36265.33999999991|        1.1936936936936937|
|         215|2020-01-01 00:00:00|       green|          36126.949999999924|        1.2023121387283238|
|         195|2020-01-01 00:00:00|       green|          13429.329999999989|         1.211726384364821|
|          88|2020-01-01 00:00:00|       green|                 

In [21]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')